## Helper classes for double stage Monte Carlo analysis

Create a `DoubleStageRocket` with all parameters of full Dauntless model (sustainer motor should not be added)

Create a `Rocket` for the sustainer and another for the booster (use `EmptyMotor` or a motor with negligible thrust and burn time)

Add either the sustainer or the booster (only one for a Monte Carlo analysis)

Create stochastic objects as usual (replace the `Rocket` for `DoubleStageRocket`)

In [ ]:
from rocketpy import Rocket

class DoubleStageRocket(Rocket):
    def __init__(
        self,
        name,
        radius,
        mass,
        inertia,
        power_off_drag,
        power_on_drag,
        center_of_mass_without_motor,
        coordinate_system_orientation="tail_to_nose",
    ):
        super().__init__(
            name,
            radius,
            mass,
            inertia,
            power_off_drag,
            power_on_drag,
            center_of_mass_without_motor,
            coordinate_system_orientation,
        )
        self.child_stage = None

    def add_child_stage(self, child_stage):
        self.child_stage = child_stage

In [ ]:
from rocketpy import MonteCarlo
from rocketpy import Flight

class DoubleStageMonteCarlo(MonteCarlo):
    def __init__(
        self,
        filename,
        environment,
        rocket,
        flight,
        export_list=None,
        data_collector=None,
    ):
        super().__init__(
            filename,
            environment,
            rocket,
            flight,
            export_list,
            data_collector,
        )

    def __run_single_simulation(self):
        env = self.environment.create_object()
        if self.rocket.child_stage is not None:
            first_stage = Flight(
                rocket=self.rocket.create_object(),
                environment=env,
                rail_length=self.flight._randomize_rail_length(),
                inclination=self.flight._randomize_inclination(),
                heading=self.flight._randomize_heading(),
                initial_solution=self.flight.initial_solution,
                terminate_on_apogee=True,
                time_overshoot=self.flight.time_overshoot,
            )
            return Flight(
                rocket=self.rocket.child_stage.create_object(),
                environment=env,
                rail_length=0,
                inclination=0,
                heading=0,
                terminate_on_apogee=self.flight.terminate_on_apogee,
                time_overshoot=self.flight.time_overshoot,
                initial_solution=first_stage,
            )
        return Flight(
            rocket=self.rocket.create_object(),
            environment=env,
            rail_length=self.flight._randomize_rail_length(),
            inclination=self.flight._randomize_inclination(),
            heading=self.flight._randomize_heading(),
            initial_solution=self.flight.initial_solution,
            terminate_on_apogee=self.flight.terminate_on_apogee,
            time_overshoot=self.flight.time_overshoot,
        )

In [ ]:
booster = Rocket(
    name="Booster",
)

In [ ]:

from rocketpy.motors import SolidMotor

sustainer = Rocket(
    name="Sustainer",
    radius=11.4 / 200,
    mass=3.703,  # rocket's mass without the motor in kg
    inertia=( 
        1.364, # (1/4) * mass * radius^2 + (1/12) * mass * length^2
        1.364, # (1/4) * mass * radius^2 + (1/12) * mass * length^2
        0.00602 # (1/2) * mass * radius^2
    ),  # in relation to the rocket's center of mass without motor
    power_off_drag="drag.csv",
    power_on_drag="drag.csv",
    center_of_mass_without_motor=0.92,
    coordinate_system_orientation="tail_to_nose",
)

nose_cone = sustainer.add_nose(length=0.572, kind="von karman", position=2.1)

fin_set = sustainer.add_trapezoidal_fins(
    n=3,
    root_chord=0.305,
    tip_chord=0.0508,
    span=0.102,
    cant_angle=0,
    sweep_length=0.254,
    airfoil=("airfoil.csv", "radians"),
    position=0,)

parachute_nosecone = sustainer.add_parachute(
    name="main",
    cd_s=1.358, # reference area * drag coefficient
    trigger="apogee",  # ejection altitude in meters
    sampling_rate=105,
    lag=0,
)

parachute_body = sustainer.add_parachute(
    name="main",
    cd_s=1.358, # reference area * drag coefficient
    trigger="apogee",  # ejection altitude in meters
    sampling_rate=105,
    lag=0,
)

motor = SolidMotor(
    thrust_source="thrust.csv",
    dry_mass=2.932,
    burn_time=3.27,
    dry_inertia=(0.2722, 0.2722, 0.0037551),
    grain_number=6,
    grain_density=1080,
    grain_outer_radius=0.0375,
    grain_initial_inner_radius=0.015,
    grain_initial_height=0.1691,
    grain_separation=0,
    grains_center_of_mass_position=0.5125,
    center_of_dry_mass_position=0.5125,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

sustainer.add_motor(motor, position=0.104)

In [ ]:
dauntless = DoubleStageRocket(
    name="Dauntless",
)

In [ ]:
dauntless.add_child_stage(sustainer)